# Mars Rover Terrain Segmentation

This notebook runs the repository end to end in a Kaggle-friendly way.

- The first code cell can either use a repo already present in Kaggle or clone it from GitHub.
- It then adds the repo folder to `sys.path` so the local packages import correctly.
- It uses the project dataset adapters, model factory, Lightning module, and evaluation flow.
- If an AI4MARS folder is mounted, it will use that data.
- Otherwise it creates a tiny offline fixture so the notebook still self-runs without network access.
- The model is built through the same code path as training; SegFormer now falls back to a small local config when pretrained weights or configs are unavailable.

For a full run on your own data, increase the batch limits and epochs in the trainer config cell.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path | None:
    search_roots = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base]
        candidates.extend(path for path in base.rglob("*") if path.is_dir())
        for candidate in candidates:
            if (candidate / "configs").exists() and (candidate / "train").exists():
                return candidate
    return None


REPO_URL = os.environ.get("MARS_SEG_REPO_URL", "https://github.com/USERNAME/REPO.git")
REPO_DIR = Path(os.environ.get("MARS_SEG_REPO_DIR", "/kaggle/working/mars-rover-terrain-segmentation"))

repo_root = find_repo_root()
if repo_root is None:
    if not REPO_DIR.exists():
        print(f"Cloning repository into {REPO_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    repo_root = REPO_DIR

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Added to sys.path: {repo_root in map(Path, sys.path)}")

In [ ]:
from __future__ import annotations

import json
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pytorch_lightning as pl
import torch
from omegaconf import OmegaConf
from PIL import Image
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

work_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else repo_root
output_dir = work_root / "outputs" / "kaggle_mars_segmentation"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {repo_root}")
print(f"Output dir: {output_dir}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
DATASET_MODE = "ai4mars"  # keep this as-is for the self-running notebook
TRAIN_LIMIT = 3
VAL_LIMIT = 2
TEST_LIMIT = 2
MAX_EPOCHS = 2


def find_ai4mars_root(search_roots: list[Path]) -> Path | None:
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base]
        candidates.extend(path for path in base.rglob("*") if path.is_dir())
        for candidate in candidates:
            has_images = (candidate / "images").exists() and any((candidate / "images").rglob("*.png"))
            has_masks = (candidate / "masks").exists() and any((candidate / "masks").rglob("*.png"))
            if has_images and has_masks:
                return candidate
    return None


def _make_sample_image(split_index: int, sample_index: int, size: int = 128) -> np.ndarray:
    grid_y, grid_x = np.mgrid[:size, :size]
    image = np.zeros((size, size, 3), dtype=np.uint8)
    image[..., 0] = (grid_x * 2 + split_index * 25 + sample_index * 40) % 255
    image[..., 1] = (grid_y * 2 + split_index * 50 + sample_index * 30) % 255
    image[..., 2] = ((grid_x + grid_y) + split_index * 15 + sample_index * 20) % 255
    return image


def _make_sample_mask(size: int = 128) -> np.ndarray:
    mask = np.zeros((size, size), dtype=np.uint8)
    half = size // 2
    mask[:half, :half] = 1
    mask[:half, half:] = 2
    mask[half:, :half] = 3
    mask[half:, half:] = 0
    return mask


def build_fixture_ai4mars_root(target_root: Path) -> Path:
    if target_root.exists():
        shutil.rmtree(target_root)
    for split in ("train", "val", "test"):
        (target_root / split).mkdir(parents=True, exist_ok=True)

    split_sizes = {"train": 3, "val": 1, "test": 1}
    for split_index, (split, count) in enumerate(split_sizes.items()):
        split_dir = target_root / split
        for sample_index in range(count):
            stem = f"{split}_{sample_index}"
            image = _make_sample_image(split_index, sample_index)
            mask = _make_sample_mask()
            Image.fromarray(image).save(split_dir / f"{stem}.png")
            Image.fromarray(mask).save(split_dir / f"{stem}_mask.png")

            if sample_index == 0:
                rng = np.zeros_like(mask, dtype=np.uint8)
                rng[:32, :32] = 1
                Image.fromarray(rng).save(split_dir / f"{stem}_rng.png")
            if split == "train" and sample_index == 1:
                rover = np.zeros_like(mask, dtype=np.uint8)
                rover[96:, 96:] = 1
                Image.fromarray(rover).save(split_dir / f"{stem}_rover.png")

    return target_root


def build_cfg(dataset_mode: str, data_root: Path | None) -> OmegaConf:
    if dataset_mode == "mars_bench":
        data_cfg = {
            "name": "mars_bench",
            "taxonomy": "core",
            "hf_dataset": "Mirali33/mb-mars_seg_msl",
            "hf_config": None,
            "image_key": "image",
            "mask_key": "mask",
            "splits": {"train": "train", "val": "validation", "test": "test"},
            "num_classes": 3,
            "ignore_index": 255,
            "loader": {"batch_size": 2, "num_workers": 0},
            "image_size": [128, 128],
            "normalize": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
            "augmentations": {
                "enabled": True,
                "rotation": 10,
                "hflip": 0.5,
                "vflip": 0.0,
                "brightness": 0.2,
                "contrast": 0.2,
                "noise": 0.02,
                "blur": 0.1,
            },
        }
    else:
        data_cfg = {
            "name": "ai4mars",
            "taxonomy": "core",
            "root": str(data_root),
            "use_rover_mask_as_label": False,
            "num_classes": 3,
            "ignore_index": 255,
            "loader": {"batch_size": 2, "num_workers": 0},
            "image_size": [128, 128],
            "normalize": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
            "augmentations": {
                "enabled": True,
                "rotation": 10,
                "hflip": 0.5,
                "vflip": 0.0,
                "brightness": 0.2,
                "contrast": 0.2,
                "noise": 0.02,
                "blur": 0.1,
            },
            "train": {"name": "train"},
            "val": {"name": "val"},
            "test": {"name": "test"},
        }

    cfg = {
        "seed": 42,
        "num_classes": 3,
        "ignore_index": 255,
        "paths": {"root": str(repo_root), "output_dir": str(output_dir)},
        "logging": {"save_visuals": True, "visuals_every_n_epochs": 1, "visuals_max_samples": 2},
        "optimizer": {"lr": 6e-5, "weight_decay": 0.01},
        "ckpt_path": None,
        "data": data_cfg,
        "model": {
            "name": "segformer_b0",
            "pretrained": False,
            "checkpoint": None,
            "loss": {"type": "ce", "dice_weight": 0.0},
        },
        "trainer": {
            "max_epochs": MAX_EPOCHS,
            "accelerator": "auto",
            "devices": 1,
            "precision": "32-true",
            "log_every_n_steps": 1,
            "check_val_every_n_epoch": 1,
            "limit_train_batches": TRAIN_LIMIT,
            "limit_val_batches": VAL_LIMIT,
            "limit_test_batches": TEST_LIMIT,
            "enable_model_summary": False,
        },
    }
    return OmegaConf.create(cfg)


existing_root = find_ai4mars_root([
    Path("/kaggle/input"),
    repo_root / "data",
    repo_root,
])
fixture_root = build_fixture_ai4mars_root(work_root / "ai4mars_fixture")

data_root = existing_root if existing_root is not None else fixture_root
print(f"Using AI4MARS root: {data_root}")
print(f"Existing mounted dataset found: {existing_root is not None}")

In [ ]:
from datasets.ai4mars import audit_ai4mars
from datasets.datamodule import SegmentationDataModule
from models.factory import build_model
from train.segmentation_module import SegmentationModule
from utils.seed import set_seed

cfg = build_cfg(DATASET_MODE, data_root)
set_seed(cfg.seed)

if cfg.data.name == "ai4mars":
    reports = [audit_ai4mars(root=cfg.data.root, split=split, taxonomy=cfg.data.taxonomy) for split in ("train", "val", "test")]
    print(json.dumps(reports, indent=2))
else:
    from datasets.mars_bench import audit_mars_bench
    reports = [
        audit_mars_bench(
            dataset_name=cfg.data.hf_dataset,
            split=cfg.data.splits[split],
            taxonomy=cfg.data.taxonomy,
            image_key=cfg.data.image_key,
            mask_key=cfg.data.mask_key,
            config_name=cfg.data.hf_config,
            max_samples=2,
        )
        for split in ("train", "val", "test")
    ]
    print(json.dumps(reports, indent=2))

data_module = SegmentationDataModule(cfg)
data_module.setup("fit")

train_batch = next(iter(data_module.train_dataloader()))
print({key: tuple(value.shape) if hasattr(value, "shape") else type(value) for key, value in train_batch.items() if key in {"image", "mask"}})
print(train_batch["mask"][0, :8, :8])

model = build_model(cfg)
module = SegmentationModule(cfg, model)
module.eval()
with torch.no_grad():
    logits = module(train_batch["image"][:1])
print(f"Logits shape: {tuple(logits.shape)}")

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=cfg.paths.output_dir,
    filename="epoch{epoch:03d}-miou{val_miou:.4f}",
    monitor="val_miou",
    mode="max",
    save_last=True,
)
lr_monitor = LearningRateMonitor(logging_interval="epoch")
logger = TensorBoardLogger(save_dir=cfg.paths.output_dir, name="lightning_logs")

trainer = pl.Trainer(
    **OmegaConf.to_container(cfg.trainer, resolve=True),
    callbacks=[checkpoint_callback, lr_monitor],
    logger=logger,
    default_root_dir=cfg.paths.output_dir,
    deterministic=True,
)

trainer.fit(module, datamodule=data_module)
print(f"Best checkpoint: {checkpoint_callback.best_model_path}")
print(f"Last checkpoint: {checkpoint_callback.last_model_path}")

In [ ]:
best_ckpt = checkpoint_callback.best_model_path or checkpoint_callback.last_model_path
if not best_ckpt:
    raise RuntimeError("No checkpoint was saved during training.")

eval_model = build_model(cfg)
eval_module = SegmentationModule(cfg, eval_model)
state = torch.load(best_ckpt, map_location="cpu")
eval_module.load_state_dict(state["state_dict"], strict=True)

eval_trainer = pl.Trainer(
    **OmegaConf.to_container(cfg.trainer, resolve=True),
    logger=False,
    enable_checkpointing=False,
    deterministic=True,
)
test_results = eval_trainer.test(eval_module, datamodule=data_module, verbose=False)
print(json.dumps(test_results, indent=2))

artifacts = sorted(path.relative_to(cfg.paths.output_dir).as_posix() for path in Path(cfg.paths.output_dir).rglob("*") if path.is_file())
print("Saved artifacts:")
for artifact in artifacts:
    print("-", artifact)